<br>
<a href="https://www.nvidia.com/en-us/training/">
    <div style="width: 55%; background-color: white; margin-top: 50px;">
    <img src="https://dli-lms.s3.amazonaws.com/assets/general/nvidia-logo.png"
         width="400"
         height="186"
         style="margin: 0px -25px -5px; width: 300px"/>
    </div>
</a>
<h1 style="line-height: 1.4;"><font color="#76b900"><b>Rapid Application Development<br>using Large Language Models</b></h1>
<h2><b>Notebook 1:</b> Getting Started With Large Language Models</h2>
<br>

**Welcome To The Course!** This is the first content notebook and is intended to springboard you into the LLM loading workflow with some insights about our problem, our resources, and our objectives!

#### **Learning Objectives:**

- Review some basic assumptions about deep learning and show how they extend to language modeling.
- Pull in your first LLM into the environment, investigate its architecture, and see how it performs!

-------

<br>

## **Part 1.1:** Recalling Deep Learning

Throughout your learning adventure with deep learning, you have probably optimized a variety of models for tasks like classification and regression. In order, you probably advanced in something like the following:

- When you started out, you used **linear and logistic regression** to model and interpret simple linear relationships that associated your inputs with your outputs.
- When that wasn't enough, you started **stacking linear layers one after another and adding non-linear activations** to give your model more predictive power.
- When your data started getting intractably high-dimensional, you started using more **informed sparsely-connected techniques like convolution** to add more control to your reasoning criteria.
- When you realized that you didn't have enough data to properly train your models for each specific task, you got **pre-trained components (i.e. VGG-16/ResNet)** that were trained on a giant repository of training data and already contained the necessary logic you wanted.

> <div><img src="imgs/machine-learning-process.jpg" width="800"/></div>
>
> **Source: [High-Performance Data Science with RAPIDS | NVIDIA](https://www.nvidia.com/en-us/deep-learning-ai/software/rapids/)**

If you’ve gone through these steps, you already possess the foundational skills to tackle complex topics, including the vast field of language modeling.

Like vision, language is highly complex and high-dimensional. For instance, a simple 200x200 color image contains $200\times 200\times 3 = 120,000$ features! Now, consider the even larger number of combinations possible in natural language. **It’s enormous!** Fortunately, creative techniques and large pre-trained models simplify the problem, providing efficient solutions.

**This course will show you how to approach language modeling, the tools available, and the types of problems you can solve!**

-------

<br>

## **Part 1.2:** Pulling In Our First LLM

Rather than building models from scratch, this course focuses on using powerful tools to simplify the process. A great place to start is **HuggingFace** &#x1F917;.

[**HuggingFace**](https://huggingface.co/) is an open-source community that offers simple strategies for accessing, developing, and hosting large deep learning models for testing and deployment. The topics they support span many tasks and modalities, but we'll be focusing on large language models (**LLMs**) for most of this course.

When searching through the [**HuggingFace Models catalog**](https://huggingface.co/models?sort=downloads&search=bert), you'll quickly stumble upon the [`bert-base-uncased`](https://huggingface.co/bert-base-uncased) model. Taking a look at its card, you'll see several interesting things:

- **The [`transformers`](https://github.com/huggingface/transformers) Package**: Loading in the model requires the use of the [`transformers`](https://github.com/huggingface/transformers) package, which is HuggingFace’s primary library for language models. Its name refers to the transformer architecture, which we’ll explore further in upcoming sections. You’ll be using `transformers` throughout this course, so feel free to dive into the source code as you go.

- **Pipelines**: HuggingFace simplifies complex deep learning tasks with its [Pipeline API](https://huggingface.co/docs/transformers/main_classes/pipelines). A pipeline allows you to move from input to output without worrying about the internal workings. We’ll demonstrate this with the `bert-base-uncased` model for mask-filling.

As a representative example, we can go ahead and pull in the discussed [`bert-base-uncased`](https://huggingface.co/bert-base-uncased) model and test it out:

In [2]:
from transformers import pipeline

## Loading in the pipeline and predict the mask fill (example from model card)
unmasker = pipeline(
    'fill-mask', 
    model='bert-base-uncased',
    device='cuda',  ## Feel free to use GPU. For such a small model, not necessary
)
unmasker("Hello I'm a [MASK] model.")

[{'score': 0.10731087625026703,
  'token': 4827,
  'token_str': 'fashion',
  'sequence': "hello i ' m a fashion model."},
 {'score': 0.08774493634700775,
  'token': 2535,
  'token_str': 'role',
  'sequence': "hello i ' m a role model."},
 {'score': 0.053383927792310715,
  'token': 2047,
  'token_str': 'new',
  'sequence': "hello i ' m a new model."},
 {'score': 0.04667212441563606,
  'token': 3565,
  'token_str': 'super',
  'sequence': "hello i ' m a super model."},
 {'score': 0.027095848694443703,
  'token': 2986,
  'token_str': 'fine',
  'sequence': "hello i ' m a fine model."}]

**And just like that, it works!** You have just obtained an intuitive answer that makes sense with human logic, which makes you almost forget that there is actually a deep learning model calculating probabilities to make this all work.

**But that's what this course is for: to see what's actually going on behind the scenes and know how to use it to make interesting and useful systems.**

<hr>
<br>

## **Part 1.3:** Dissecting The Pipeline

While the pipeline provides a clean interface (taking strings in and returning a dictionary), it doesn’t give us much insight into what’s happening under the hood. Let’s peel back a layer and examine the approximate inner workings:

In [1]:
from transformers import AutoTokenizer, BertTokenizer, BertModel, FillMaskPipeline, AutoModelForMaskedLM, BertForMaskedLM, BertForPreTraining 

from transformers import AutoTokenizer, AutoModel        ## General-purpose fully-automatic
from transformers import AutoModelForMaskedLM            ## Default import for FillMaskPipeline
from transformers import BertTokenizer, BertForMaskedLM  ## Realized components after automatic resolution

class MyMlmPipeline(FillMaskPipeline):
    ## My Masked Language Modeling Pipeline

    ### CASE 0: Construct your pipeline automatically by pulling in the components
    ###   with their respective configs from HuggingFace. Pipeline assumes preprocessing/postprocessing.
    def __init__(self, *args, **kwargs):
        ## The fully-automatic version
        super().__init__(
            tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased'),
            model = AutoModelForMaskedLM.from_pretrained("bert-base-uncased"),
            # model = BertForMaskedLM.from_pretrained("bert-base-uncased"),
            *args, **kwargs  ## <- pass in any extra arguments
        )

    ### CASE 1: Uncomment out the __call__ method to see what data is flowing.
    def __call__(self, string, verbose=True):
        ## Verbose argument just there for our convenience
        input_tensors = self.preprocess(string)
        if verbose: print('\npreprocess outputs:\n', input_tensors, '\n')
        output_tensors = self.forward(input_tensors)
        if verbose: print('forward outputs:\n', output_tensors, '\n')
        output = self.postprocess(output_tensors)
        return output
    
    ### CASE 2: Uncomment out the manual overrides below to verify the pipeline still works
    def preprocess(self, string):
        string = [string] if isinstance(string, str) else string
        inputs = self.tokenizer(string, return_tensors="pt")           ### strings -> indices
        inputs = {k: v.to("cuda") for k, v in inputs.items()}          ### move to GPU
        return inputs

    def forward(self, tensor_dict):
        output_tensors = self.model.forward(**tensor_dict)             ### indices -> vectors -> probabilities
        return {**output_tensors, **tensor_dict}

    def postprocess(self, tensor_dict):
        ## Very Task-specific; see FillMaskPipeline.postprocess
        tensor_dict = {k: v.to("cpu") for k, v in tensor_dict.items()} ### move off GPU
        return super().postprocess(tensor_dict)                        ### probabilities (or vectors) -> outputs
        
unmasker = MyMlmPipeline(device="cuda")
unmasker("Hello, Mr. Bert! How is it [MASK]?")

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]


preprocess outputs:
 {'input_ids': tensor([[  101,  7592,  1010,  2720,  1012, 14324,   999,  2129,  2003,  2009,
           103,  1029,   102]], device='cuda:0'), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]], device='cuda:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]], device='cuda:0')} 

forward outputs:
 {'logits': tensor([[[ -6.6978,  -6.6436,  -6.6542,  ...,  -5.9664,  -5.8454,  -4.0651],
         [ -7.1922,  -7.0817,  -7.2808,  ...,  -6.6275,  -6.7937,  -5.0987],
         [-14.1895, -14.3331, -14.3951,  ..., -11.4550, -10.5485, -10.7081],
         ...,
         [ -7.8704,  -7.9647,  -7.8207,  ...,  -6.7212,  -7.0204,  -4.0526],
         [-10.7055, -10.2556, -10.7905,  ...,  -9.1913, -10.1996,  -0.9692],
         [-10.9574, -11.0610, -10.8301,  ...,  -9.3834,  -9.7457,  -8.4909]]],
       device='cuda:0', grad_fn=<ViewBackward0>), 'input_ids': tensor([[  101,  7592,  1010,  2720,  1012, 14324,   999,  2129,  2003,  2009,
           1

[{'score': 0.23865845799446106,
  'token': 2183,
  'token_str': 'going',
  'sequence': 'hello, mr. bert! how is it going?'},
 {'score': 0.0717877447605133,
  'token': 2017,
  'token_str': 'you',
  'sequence': 'hello, mr. bert! how is it you?'},
 {'score': 0.05827954038977623,
  'token': 6230,
  'token_str': 'happening',
  'sequence': 'hello, mr. bert! how is it happening?'},
 {'score': 0.0563340038061142,
  'token': 2651,
  'token_str': 'today',
  'sequence': 'hello, mr. bert! how is it today?'},
 {'score': 0.05287010222673416,
  'token': 2085,
  'token_str': 'now',
  'sequence': 'hello, mr. bert! how is it now?'}]

We can also see that the model primarily consists of two core components:
- **Tokenizer**: Converts input strings into a format the model can process.
- **Model**: The deep learning engine that transforms input tensors into output tensors.

Together, these components support the pipeline’s intuitive workflow:
- `preprocess`: human-intuitive input $\to$ tensor inputs. Facilitated by `tokenizer`
- `forward`: tensor inputs $\to$ tensor outputs. Facilitated by `model`
- `postprocess`: tensor outputs $\to$ human-intuitive outputs. Facilitated by the pipeline task.

For deep learning, this flow makes sense. The model operates on numbers, but the users are working with words, to this abstraction greatly simplifies the LLM interface for typical scenarios.

-------

<br>

## **Part 1.4:** Your Course Environment

Since these models are so easy to pull in, we need to establish some limitations! Throughout this course, feel free to explore different models, but be mindful of their size. **Your compute environment is powerful, but not limitless.** We’ve pre-loaded several models for you, which you can review (along with their licenses) in the [**extras_and_licenses/99_licenses.ipynb**](extras_and_licenses/99_licenses.ipynb).

For this course, you’ll be using a high-performance compute setup which is sufficient for the exercises and has the following key resources:

- **System Memory**: Large language models can require massive memory, with some needing tens or even hundreds of GB. This environment is equipped to handle significant workloads, but try to keep model sizes reasonable unless otherwise instructed.
  
- **GPU**: GPU acceleration is crucial for fast deep learning, especially given the parallel processing demands of deep models. The thousands of [**CUDA cores**](https://en.wikipedia.org/wiki/CUDA) in modern NVIDIA GPUs significantly speed up training and inference by performing parallelizable tasks (like high-dimensional matrix multiplication) very quickly.
  
    - **GPU RAM**: Large models need to reside in GPU memory for fast inference. Limited GPU RAM can be a bottleneck for using accelerated LLMs, so manage your model sizes accordingly.

Here’s a breakdown of your allocated resources:

In [3]:
%%bash
echo """
===================================================
GPU SPECIFICATION
==================================================="""
nvidia-smi
echo """
===================================================
MEMORY SPECIFICATION
==================================================="""
free -h


GPU SPECIFICATION
Thu Oct  1 19:00:03 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.86.15              Driver Version: 570.86.15      CUDA Version: 13.0     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100 80GB PCIe          On  |   00000001:00:00.0 Off |                    0 |
| N/A   39C    P0             67W /  300W |   49497MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+----------------------------

<br>

**So, while you have a solid compute budget, it’s not infinite!**

Before moving to the next notebook, restart the Jupyter kernel using the following code to clear memory and avoid issues with future notebooks:

In [4]:
import IPython
app = IPython.Application.instance()
app.kernel.do_shutdown(True)

/tmp/ipykernel_3773/1401414292.py:2: DeprecationWarning: `IPython.Application` is only a re-export of `traitlets.config.application.Application`; import it from traitlets directly. Accessing it here triggers an import of `IPython.core.application`, which is no longer imported when IPython is -- import that module explicitly if you rely on that import happening, in particular if you also rely on other submodules being transitively imported as a side effect.
  app = IPython.Application.instance()


{'status': 'ok', 'restart': True}

<hr>
<br>

# <font color="#76b900">**Wrapping Up**</font>

Now that you’ve seen how easy it is to load a model, it’s time to tackle some critical questions:

**Can I actually use these models?** That really depends on licensing, which you can read more about in [`99_licenses.ipynb`](extras_and_licenses/99_licenses.ipynb):

- ***The earlier models we'll look at will have licenses associated with the data, and many of the fine-tuned ones are trained for proof of concept only and are therefore not commercially-viable.*** After this course, you'll be able to experiment with them and see whether you can find one that's good enough and also viable. Alternatively, you'll be able to take some inspiration and maybe consider fine-tuning an architecture on your own dataset!

- ***The later models we'll look at are viable for use commercially and are extremely powerful and general!*** They are amazing on their own and can work great with the smaller models to satisfy compute budgets and control structures! Furthermore, they naturally interop with more powerful variations with expanded feature sets and emergent capabilities.

**How and why do they work?** This will be talked about at length.

**What models do you choose?** This will be talked about at length. 

**In the next few notebooks, we'll be getting familiar with how these systems work!**

<font color="#76b900">**Get excited, and welcome to the course!!**</font>

<a href="https://www.nvidia.com/en-us/training/">
    <div style="width: 55%; background-color: white; margin-top: 50px;">
    <img src="https://dli-lms.s3.amazonaws.com/assets/general/nvidia-logo.png"
         width="400"
         height="186"
         style="margin: 0px -25px -5px; width: 300px"/>
    </div>
</a>